# BEE 4750 Homework 3: Dissolved Oxygen and Monte Carlo

**Name**: Eveyln Chen and Natalie Ho 

**ID**: , nh424

> **Due Date**
>
> Thursday, 10/16/25, 9:00pm

## Overview

### Instructions

-   Problem 1 asks you to implement a model for dissolved oxygen in a
    river with multiple waste releases and use this to develop a
    strategy to ensure regulatory compliance.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/hw2-natalie/lab-2-NatalieHo424/hw3-natalie-evelyn`
   Installed PrettyTables ─ v3.0.11
Precompiling project...
  19856.5 ms  ✓ PrettyTables
  27233.9 ms  ✓ DataFrames
   1998.0 ms  ✓ Latexify → DataFramesExt
  3 dependencies successfully precompiled in 50 seconds. 211 already precompiled.


In [2]:
using Random
using Plots
using LaTeXStrings
using Distributions

## Problems (Total: 30 Points)

### Problem 1 (30 points)

A river which flows at 6 km/d is receiving waste discharges from two
sources which are 15 km apart. The oxygen reaeration rate is 0.55
day<sup>-1</sup>, and the decay rates of CBOD and NBOD are are 0.35 and
0.25 day<sup>-1</sup>, respectively. The river’s saturated dissolved
oxygen concentration is 10m g/L.

If the characteristics of the river inflow and waste discharges are
given in <a href="#tbl-river" class="quarto-xref">Table 1</a>, write a
Julia model to compute the dissolved oxygen concentration from the first
wastewater discharge to an arbitrary distance `d` km downstream. Use
your model to compute the minimum dissolved oxygen concentration up to
50 km downstream and how far downriver this maximum occurs.

| Parameter | River Inflow | Waste Stream 1 | Waste Stream 2 |
|:--:|---:|---:|---:|
| Inflow | 100,000 m<sup>3</sup>/d | 10,000 m<sup>3</sup>/d | 15,000 m<sup>3</sup>/d |
| DO Concentration | 7.5 mg/L | 5 mg/L | 5 mg/L |
| CBOD | 5 mg/L | 50 mg/L | 45 mg/L |
| NBOD | 5 mg/L | 35 mg/L | 35 mg/L |

Table 1: River inflow and waste stream characteristics for Problem 1.

### Problem 1.1

Implement the Streeter-Phelps (analytic) solution for the dissolved
oxygen concentration. Plot the dissolved oxygen concentration from the
first waste stream to 50 km downriver. What is the minimum value in
mg/L?

In [41]:
using Plots
using LaTeXStrings

function do_simulate(x, C0, B0, N0, ka, kc, kn, Cs, U)
    B = B0 * exp(-kc * x / U)
    N = N0 * exp(-kn * x / U)

    α1 = exp(-ka * x / U)
    α2 = (kc/(ka-kc)) * (exp(-kc * x / U) - exp(-ka * x / U))
    α3 = (kn/(ka-kn)) * (exp(-kn * x / U) - exp(-ka * x / U))

    C = Cs * (1 - α1) + (C0 * α1) - (B0 * α2) - (N0 * α3)
    return (C, B, N)
end

U, ka, kc, kn = 6.0, 0.55, 0.35, 0.25
Cs = 10.0

Qr, Q1, Q2 = 100_000.0, 10_000.0, 15_000.0
Cr, Br, Nr = 7.5, 5.0, 5.0
Cw1, Bw1, Nw1 = 5.0, 50.0, 35.0
Cw2, Bw2, Nw2 = 5.0, 45.0, 35.0

x_split, x_end, Δx = 15.0, 50.0, 0.05
x1      = 0.0:Δx:x_split
x2local = 0.0:Δx:(x_end - x_split)

# ---------- Reach 1: mix river + Waste 1 at x = 0 ----------
Qtot1 = Qr + Q1
C01 = (Qr*Cr + Q1*Cw1) / Qtot1
B01 = (Qr*Br + Q1*Bw1) / Qtot1
N01 = (Qr*Nr + Q1*Nw1) / Qtot1

do_out1 = (ξ -> do_simulate(ξ, C01, B01, N01, ka, kc, kn, Cs, U)).(x1)
C1 = [d[1] for d in do_out1]
B1 = [d[2] for d in do_out1]
N1 = [d[3] for d in do_out1]

C15m = C1[end]
B15m = B01 * exp(-kc * x_split / U)
N15m = N01 * exp(-kn * x_split / U)

# ---------- Reach 2: mix with Waste 2 at x = 15 ----------
Qtot2 = Qtot1 + Q2
C02 = (Qtot1*C15m + Q2*Cw2) / Qtot2
B02 = (Qtot1*B15m + Q2*Bw2) / Qtot2
N02 = (Qtot1*N15m + Q2*Nw2) / Qtot2

do_out2 = (ξ -> do_simulate(ξ, C02, B02, N02, ka, kc, kn, Cs, U)).(x2local)
C2 = [d[1] for d in do_out2]
B2 = [d[2] for d in do_out2]
N2 = [d[3] for d in do_out2]

# ---------- Concatenate full profile ----------
x = vcat(collect(x1), x_split .+ collect(x2local))
C = vcat(C1, C2)
B = vcat(B1, B2)
N = vcat(N1, N2)

# ---------- Minimum DO ----------
imin = argmin(C)
xmin = x[imin]
Cmin = C[imin]
println("Minimum DO = $(round(Cmin,digits=3)) mg/L at x = $(round(xmin,digits=2)) km")

p1 = plot(
    x1, C1; color=:black, lw=3, label="DO (reach 1)",
    xlabel="Distance (km)", ylabel="DO/OD (mg/L)",
    size=(1000, 380), dpi=150, guidefontsize=14, tickfontsize=11, legendfontsize=11
)
plot!(p1, x_split .+ x2local, C2; color=:black, lw=3, ls=:solid, label="DO (reach 2)")

plot!(p1, x1, B1; color=:green,  ls=:dash, lw=2, label="CBOD (r1)")
plot!(p1, x_split .+ x2local, B2; color=:green, ls=:dash, lw=2, label="CBOD (r2)")
plot!(p1, x1, N1; color=:blue,   ls=:dash, lw=2, label="NBOD (r1)")
plot!(p1, x_split .+ x2local, N2; color=:blue,  ls=:dash, lw=2, label="NBOD (r2)")

plot!(p1, [0, x_end], [Cs, Cs]; color=:purple, ls=:dot, lw=2, label=L"C_s")
vline!([x_split]; color=:gray, ls=:dash, label="Waste 2 @ $(x_split) km")

xlims!(p1, 0, x_end)
ylims!(p1, 0, Cs + 1)

display(p1)

Minimum DO = 3.756 mg/L at x = 22.35 km


### Problem 1.2

Implement a numerically-integrated (discretized) solution for the
dissolved oxygen concentration. Using a resolution of 0.5km, conduct a
simulation and plot the results on the same axis as the plot from
Problem 1.1. How has the minimum value changed? What do you attribute
this difference to (be specific about the source of the difference(s) in
terms of the simulation dynamics, not just that one is a numerical
approximation).

### Problem 1.3

Using the analytic model, what is the minimum level of treatment (%
removal of organic waste; assume this is equivalent to the same level of
reduction of the CBOD and NBOD) for waste stream 1 that will ensure that
the dissolved oxygen concentration is in compliance with the 4 mg/L
standard along with a 5% margin of safety, assuming that waste stream 2
remains untreated? How about if only waste stream 2 is treated?

### Problem 1.4

Suppose you are responsible for designing a waste treatment plan for
discharges into the river, with a regulatory mandate to keep the
dissolved oxygen concentration above 4 mg/L. Discuss whether you’d opt
to treat waste stream 2 alone or both waste streams equally. What other
information might you need to make a conclusion, if any?

### Problem 1.5

Suppose that it is known that the DO concentrations at the river inflow
can vary according to a $\text{LogNormal}(2.0, 0.15)$ distribution.
Conduct a Monte Carlo simulation of the DO concentration in the river.
If you treat only waste stream 1 (based on your analysis from Problem
1.3), what is the expected probability and 95% confidence interval that
the river fails to comply with the regulatory standard of 4 mg/L? How
did you decide that your Monte Carlo sample size was sufficiently large?

## References

List any external references consulted, including classmates.